# Validate quality (job task)

Runs after every pipeline refresh in the job from `04_job_and_alerts`. Each check compares one number with a threshold the data owner agreed to, writes a row to `quality_results`, and **any FAIL raises an error**, which fails the job and sends its failure notifications. Change thresholds here, in one place, and record why.

In [ ]:
import importlib
import sys
from pathlib import Path

dbutils.widgets.text("run_id", "manual")
dbutils.widgets.text("course_root", "")       # set when the job is deployed from a bundle

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = Path(dbutils.widgets.get("course_root")) if dbutils.widgets.get("course_root") else _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)
N = ent.names()
RUN_ID = dbutils.widgets.get("run_id")
print(f"Validating {N['fq']} for run {RUN_ID}")

In [ ]:
# Thresholds: the data owner's decisions. Shares are fractions of the records checked.
THRESHOLDS = {
    "coded value not in a crosswalk": 0.02,     # per column
    "value didn't convert": 0.01,               # per column
    "offense and violence codes agree": 0.005,
    "birth date is plausible": 0.001,
    "custody level missing": 0.05,
}

In [ ]:
fq = N["fq"]
results = []

def record(check_name, value, threshold, ok, detail=""):
    results.append((RUN_ID, check_name, float(value), str(threshold), "PASS" if ok else "FAIL", detail))

ent.sql(f"""CREATE TABLE IF NOT EXISTS {fq}.quarantine_reviews (
    source_file STRING, tdcj_number_as_written STRING, decision STRING, reviewed_by STRING, reviewed_at TIMESTAMP)
    COMMENT 'Data owner decision for each quarantined record'""", quiet=True)

counts = spark.sql(f"""
    SELECT (SELECT count(*) FROM {fq}.bronze_offenders) AS bronze,
           (SELECT count(*) FROM {fq}.silver_offenders) AS silver,
           (SELECT count(*) FROM {fq}.silver_quarantine) AS quarantine,
           (SELECT count(*) FROM (SELECT tdcj_number FROM {fq}.silver_offenders GROUP BY 1 HAVING count(*) > 1)) AS duplicates,
           (SELECT count(*) FROM {fq}.silver_quarantine q LEFT ANTI JOIN {fq}.quarantine_reviews r
              ON r.source_file = q._source_file AND r.tdcj_number_as_written = q.tdcj_number) AS unreviewed,
           (SELECT count_if(custody_level IS NULL) / count(*) FROM {fq}.silver_offenders) AS custody_missing
""").first()

record("no records lost (bronze = silver + quarantine)", counts.bronze - counts.silver - counts.quarantine, 0,
       counts.bronze == counts.silver + counts.quarantine,
       f"bronze {counts.bronze:,}, silver {counts.silver:,}, quarantine {counts.quarantine:,}")
record("TDCJ number unique in silver", counts.duplicates, 0, counts.duplicates == 0, f"{counts.duplicates} repeated")
record("quarantined records reviewed", counts.unreviewed, 0, counts.unreviewed == 0,
       f"{counts.unreviewed} held records have no decision in quarantine_reviews")
record("custody level missing", counts.custody_missing, THRESHOLDS["custody level missing"],
       counts.custody_missing <= THRESHOLDS["custody level missing"])

for row in spark.table(f"{fq}.gold_quality_summary").collect():
    if row.check_name.endswith("in the crosswalk") or row.check_name.startswith(("offense code is", "violence code is")):
        limit = THRESHOLDS["coded value not in a crosswalk"]
    elif row.check_name.endswith("converted"):
        limit = THRESHOLDS["value didn't convert"]
    else:
        limit = THRESHOLDS.get(row.check_name, 0.01)
    record(row.check_name, row.failing_share, limit, row.failing_share <= limit, f"{row.failing_records:,} records")

table = spark.createDataFrame(results, "run_id STRING, check_name STRING, value DOUBLE, threshold STRING, status STRING, detail STRING")
from pyspark.sql import functions as F
(table.withColumn("checked_at", F.current_timestamp())
      .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{fq}.quality_results"))
display(table.orderBy("status", "check_name"))

In [ ]:
failed = [r for r in results if r[4] == "FAIL"]
if failed:
    raise AssertionError(f"{len(failed)} data-quality check(s) failed: " + "; ".join(f"{r[1]} ({r[5] or r[2]})" for r in failed))
print(f"All {len(results)} checks passed.")